In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score






# load dataset
path = "./../project/Dataset/BOM.csv"
df = pd.read_csv(path)
print(df.head(2))


         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity9am  Humidity3pm  \
0           W           44.0          W  ...        71.0         22.0   
1         WNW           44.0        NNW  ...        44.0         25.0   

   Pressure9am  Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  RainToday  \
0       1007.7       1007.1       8.0       NaN     16.9     21.8         No   
1       1010.6       1007.8       NaN       NaN     17.2     24.3         No   

   RainTomorrow  
0            No  
1            No  

[2 rows x 23 columns]


In [23]:
df = df.dropna(how='all')
# rename columns to integers
df.columns = range(df.shape[1])
# drop first column if it's name/location
df = df.drop(columns=[0])

# find target column (last column)
target_col = df.columns[-1]
# drop rows where target is NaN
df = df[df[target_col].notna()]
print(df.head(2))


       1     2     3    4   5   6    7     8    9    10  ...    13    14  \
0  Albury  13.4  22.9  0.6 NaN NaN    W  44.0    W  WNW  ...  71.0  22.0   
1  Albury   7.4  25.1  0.0 NaN NaN  WNW  44.0  NNW  WSW  ...  44.0  25.0   

       15      16   17  18    19    20  21  22  
0  1007.7  1007.1  8.0 NaN  16.9  21.8  No  No  
1  1010.6  1007.8  NaN NaN  17.2  24.3  No  No  

[2 rows x 22 columns]


In [24]:
# separate target
Y = df.pop(target_col)
Y = Y.map({'No':0, 'Yes':1})

X = df

# separate numeric and categorical columns
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# impute missing values
X[num_cols] = SimpleImputer(strategy='mean').fit_transform(X[num_cols])
X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])

# one-hot encode categorical columns
X = pd.get_dummies(X, drop_first=True)
X.columns = X.columns.astype(str)

print(X.head(2))


      2     3    4         5         6     8    11    12    13    14  ...  \
0  13.4  22.9  0.6  5.469824  7.624853  44.0  20.0  24.0  71.0  22.0  ...   
1   7.4  25.1  0.0  5.469824  7.624853  44.0   4.0  22.0  44.0  25.0  ...   

   10_NW   10_S  10_SE  10_SSE  10_SSW  10_SW   10_W  10_WNW  10_WSW  21_Yes  
0  False  False  False   False   False  False  False    True   False   False  
1  False  False  False   False   False  False  False   False    True   False  

[2 rows x 110 columns]


In [25]:
# scale features
scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# train-test split
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)



In [26]:




# try different k for KNN
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
best_k = None
best_score = -float('inf')

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(x_train, y_train)
    y_pred = knn.predict(x_test)
    score = r2_score(y_test, y_pred)
    print(f"k={k}, R^2 score={score:.4f}")
    
    if score > best_score:
        best_score = score
        best_k = k

print("\n best k:", best_k)
print(" best R^2 score:", best_score)

k=1, R^2 score=-0.3870
k=3, R^2 score=0.0515
k=5, R^2 score=0.1396
k=7, R^2 score=0.1784
k=9, R^2 score=0.1959
k=11, R^2 score=0.2057
k=13, R^2 score=0.2136
k=15, R^2 score=0.2196

 best k: 15
 best R^2 score: 0.2195878208832538
